In [1]:
import pandas as pd, numpy as np, geopandas as gpd
import matplotlib.pyplot as plt, os, sys
import networkx as nx, math, glob
import pickle, copy

import torch
from torch_geometric.data import Data
from torch_geometric.utils.convert import from_networkx, to_networkx

from torch_geometric.data import Data
from torch_geometric.utils import degree, coalesce

import scipy.sparse as sp
from torch_geometric.utils import to_scipy_sparse_matrix

from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

In [2]:
with open("motorable_ky_aadt_dual_graph_nbc2.pkl", "rb") as f:
    rd_dual_graph = pickle.load(f)

In [3]:
nodeslist = list(rd_dual_graph.nodes)
if all(isinstance(n, int) for n in nodeslist):
    sorted_nodes = sorted(nodeslist)
    is_contiguous = sorted_nodes == list(range(len(nodeslist)))
    print("Are node IDs contiguous from 0 to N-1?:", is_contiguous) #required by pytorch geometric
else:
    print("Not all node IDs are integers — mapping needed.") #required

Not all node IDs are integers — mapping needed.


In [4]:
original_nodes = sorted(list(rd_dual_graph.nodes))
uid_to_index = {uid: i for i, uid in enumerate(original_nodes)}
index_to_uid = {i: uid for uid, i in uid_to_index.items()}

nx_dgraph_indexed = nx.DiGraph()

for uid, attr in rd_dual_graph.nodes(data=True):
    nx_dgraph_indexed.add_node(uid_to_index[uid], **attr)

for u, v, edge_attrs in rd_dual_graph.edges(data=True):
        nx_dgraph_indexed.add_edge(uid_to_index[u], uid_to_index[v], **edge_attrs)

In [5]:
edge_list = list(nx_dgraph_indexed.edges())
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

In [6]:
attr_keys = list(next(iter(nx_dgraph_indexed.nodes(data=True)))[1].keys())
node_attr_lists = {key: [] for key in attr_keys}

for node_id in range(len(nx_dgraph_indexed.nodes)):
    for key in attr_keys:
        node_attr_lists[key].append(nx_dgraph_indexed.nodes[node_id].get(key, None))

In [7]:
pyg_data = Data(edge_index=edge_index)
for key, val_list in node_attr_lists.items():
        pyg_data[key] = val_list

In [ ]:
def remove_isolated_nodes(data: Data):
    # Identify isolated nodes
    src, dst = data.edge_index
    deg_src = degree(src, data.num_nodes)
    deg_dst = degree(dst, data.num_nodes)
    connected_mask = (deg_src + deg_dst) > 0  # keep only connected nodes

    # Build mapping from old index → new index
    old_to_new = {}
    new_index = 0
    for old_idx, keep in enumerate(connected_mask):
        if keep:
            old_to_new[old_idx] = new_index
            new_index += 1

    # Filter node attributes (stored as lists)
    new_data = Data()
    for key, val in data.items():
        if key == 'edge_index':
            continue  # handled below
        if isinstance(val, list) and len(val) == data.num_nodes:
            new_data[key] = [val[i] for i in range(data.num_nodes) if connected_mask[i]]
        else:
            new_data[key] = val  # keep as-is

    # Filter edges and remap node indices
    new_edges = []
    for u, v in data.edge_index.t().tolist():
        if connected_mask[u] and connected_mask[v]:
            new_edges.append((old_to_new[u], old_to_new[v]))

    new_data.edge_index = torch.tensor(new_edges, dtype=torch.long).t().contiguous()

    return new_data

In [9]:
def get_isolated_nodes_directed(data):
    # Compute in-degree and out-degree
    row, col = data.edge_index
    out_deg = degree(row, num_nodes=data.num_nodes)
    in_deg = degree(col, num_nodes=data.num_nodes)
    
    # Find nodes with both in-degree and out-degree = 0
    isolated_nodes = torch.where((out_deg == 0) & (in_deg == 0))[0]
    
    return isolated_nodes

In [10]:
#Before: check no isolated nodes exist
isolated = get_isolated_nodes_directed(pyg_data)
print(len(isolated))
isolated

144


tensor([  2190,  12020,  14929,  15149,  16414,  17444,  20419,  35832,  36149,
         36163,  38514,  38577,  41531,  43658,  51347,  55527,  61347,  66829,
         75346,  81661,  81802,  81885,  83586,  86549,  96580,  99554, 112831,
        115966, 117203, 117857, 118821, 123612, 125421, 126138, 127473, 128745,
        129154, 130432, 130476, 135755, 137747, 138194, 141278, 144006, 144204,
        144362, 144444, 148944, 155288, 155345, 156172, 157663, 161570, 162012,
        162307, 166067, 176117, 189972, 190020, 197848, 204979, 208463, 210618,
        224338, 225899, 226082, 228922, 230326, 231329, 232870, 233617, 236321,
        236400, 236563, 237475, 239822, 240744, 241157, 241292, 241400, 241480,
        244656, 247966, 248008, 249320, 249733, 250937, 251940, 254790, 255952,
        261004, 261894, 274960, 275139, 291754, 292629, 294985, 299087, 300171,
        301802, 305583, 305584, 306136, 307517, 310311, 311494, 312848, 313982,
        321666, 323514, 328006, 328352, 

In [11]:
pyg_data = remove_isolated_nodes(pyg_data)
pyg_data.edge_index = coalesce(pyg_data.edge_index, num_nodes=pyg_data.num_nodes)

In [12]:
#After: confirm no isolated nodes exist
isolated = get_isolated_nodes_directed(pyg_data)
print(len(isolated))
isolated

0


tensor([], dtype=torch.int64)

In [13]:
if pyg_data.validate():
    print("pyg_data object is properly defined")
print(f'Num node features: {pyg_data.num_node_features}')
print(f'Num edge features: {pyg_data.num_edge_features}')
print(f'Edge atributes: {pyg_data.edge_attrs()}')
print(f'Num node types (should be one; all nodes represent roadway links): {pyg_data.num_node_types}')
print(f'Does Graph contains isolated nodes?: {pyg_data.has_isolated_nodes()}')
print(f'Does Graph contains self loops?: {pyg_data.has_self_loops()}')
print(f'Graph directed?: {pyg_data.is_directed()}')
print(f'Graph edges coalesced: {pyg_data.is_coalesced()}')

pyg_data object is properly defined
Num node features: 0
Num edge features: 0
Edge atributes: ['edge_index']
Num node types (should be one; all nodes represent roadway links): 1
Does Graph contains isolated nodes?: False
Does Graph contains self loops?: False
Graph directed?: True
Graph edges coalesced: True


In [14]:
datacols = ['CO_NAME', 'TYPE_OP','AADT', 'Truck_AADT', 'Length','Mid_Lat', 'Mid_Long', 
'pop5min', 'DPop5min', 'Worker5min', 'Resident5min', 'HH5min', 'PerCapitaIncome5min', 
    'MedianAge5min', 'Employed5min', 'MdHHIn5min', 'Transport_MarterialMoving5min', 'Businesses5min', 
    'Employees5min', 'AgMn5min', 'Mining5min', 'Utilities5min', 'Construction5min', 'Manufacturing5min', 
    'Transportation_Warehousing5min', 'RlvntBusi5min',
'AvgSpeed', 'StdSpeed', 'Pcnt5Speed', 'Pcnt20Speed', 
'Pcnt50Speed', 'Pcnt85Speed', 'RURAL_URBAN', 'F_SYSTEM', 'INTERCHANGES', 'AT_GRADE_OTHER', 
'AT_GRADE_SIGNALS', 'AT_GRADE_SIGNS', 'Truck_Perc', 'TYPE_TERRAIN', 'THROUGH_LANES',
'LANESCRD','LANESNC', 'LANE_WIDTH', 'MEDIAN_TYPE', 'MEDIAN_WIDTH', 'SHLD_WIDTH_R',
'SHLD_WIDTH_L', 'ACCESS_CONTROL', 'SPEED_LIMIT_LWA', 'bcn_rac','zBCn_PR_ac', 'AADP', 'TT_BC']

In [15]:
pyg_data['Type_Operation'] = [
    'one-way highway' if val == '1' else
    'two-way highway' if val == '2' else
    'divided by median highway' if val == 'D' else val
    for val in pyg_data['TYPE_OP']
]

terrain_mapping = {
    1: 'Flat',
    2: 'Rolling',
    3: 'Mountainous'
}
pyg_data['Terrain_Type'] = [terrain_mapping.get(val, val) for val in pyg_data['TYPE_TERRAIN']]

median_mapping = {
    1: 'Concrete Barrier',
    2: 'Guardrail Barrier',
    3: 'Other Positive Barrier',
    4: 'Raised Non Mountable',
    5: 'Raised Mountable',
    6: 'Flush',
    7: 'Depressed',
    8: 'No Median'
}

pyg_data['MedianType'] = [median_mapping.get(val, val) for val in pyg_data['MEDIAN_TYPE']]

access_mapping = {
    1: 'Full',
    2: 'Partial',
    3: 'By Permit'
}

pyg_data['AccessControl'] = [access_mapping.get(val, val) for val in pyg_data['ACCESS_CONTROL']]

In [16]:
continuous_features = ['AADP', 'THROUGH_LANES', 'LANE_WIDTH','SHLD_WIDTH_R', 'SPEED_LIMIT_LWA', 
                       'F_SYSTEM', 'MdHHIn5min', 'pop5min', 'DPop5min', 'HH5min', 'Businesses5min',
                        'Worker5min', 'RlvntBusi5min', 'Resident5min',
                       'Employed5min', 'AvgSpeed', 'Pcnt85Speed',
                       'TT_BC']

In [17]:
pyg_data.x = torch.stack([
    torch.tensor(pyg_data[col], dtype=torch.float) 
    for col in continuous_features
], dim=1)
print(pyg_data.x.size())

torch.Size([404070, 18])


In [18]:
pyg_data.y = torch.tensor(pyg_data['AADT'], dtype=torch.float).unsqueeze(1)
print(pyg_data.y.size())

torch.Size([404070, 1])


In [19]:
y_nonzero = pyg_data.y[pyg_data.y > 0].squeeze()

# Sort for percentile lookup
y_sorted, _ = torch.sort(y_nonzero)

n = y_sorted.numel()
percentile = lambda p: y_sorted[int(p * n)]

describe_dict = {
    'count': y_nonzero.numel(),
    'mean': y_nonzero.mean().item(),
    'std': y_nonzero.std(unbiased=False).item(),  # match pandas (ddof=0)
    'min': y_sorted[0].item(),
    '1%': percentile(0.01).item(),
    '2.5%': percentile(0.025).item(),
    '5%': percentile(0.05).item(),
    '25%': percentile(0.25).item(),
    '50%': y_nonzero.median().item(),
    '75%': percentile(0.75).item(),
    '95%': percentile(0.95).item(),
    '97.5%': percentile(0.975).item(),
    '99%': percentile(0.99).item(),
    'max': y_sorted[-1].item()
}

describe_aadt = pd.DataFrame(describe_dict, index=['AADT'])
describe_aadt

,count,mean,std,min,1%,2.5%,5%,25%,50%,75%,95%,97.5%,99%,max
AADT,110789,5853.708496,11087.919922,1.0,38.0,71.0,114.0,594.0,2329.0,6781.0,21901.0,31726.0,47984.0,196929.0


In [20]:
pcnt_value = describe_dict['1%']
num_below = (y_nonzero < pcnt_value).sum().item()
print(f"Number of nodes with AADT < 1th percentile ({pcnt_value:,.2f}): {num_below}")

Number of nodes with AADT < 1th percentile (38.00): 1096


In [21]:
p99_value = describe_dict['99%']
num_above_99 = (y_nonzero > p99_value).sum().item()
print(f"Number of nodes with AADT > 99th percentile ({p99_value:,.2f}): {num_above_99}")

Number of nodes with AADT > 99th percentile (47,984.00): 1107


In [22]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Convert to numpy
X_np = pyg_data.x.numpy()

# Fit and transform 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

# Replace in graph
pyg_data.x = torch.tensor(X_scaled, dtype=torch.float)
print(pyg_data.x.size())

torch.Size([404070, 18])


In [23]:
cat_cols  = ['Type_Operation','RURAL_URBAN','Terrain_Type','MedianType','AccessControl', ] #'CO_NAME'
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    pyg_data[col+'_encoded'] = le.fit_transform(pyg_data[col])
    encoders[col] = le  # Save for inverse transform later if needed

In [24]:
print(encoders)
encoders['Type_Operation']

{'Type_Operation': LabelEncoder(), 'RURAL_URBAN': LabelEncoder(), 'Terrain_Type': LabelEncoder(), 'MedianType': LabelEncoder(), 'AccessControl': LabelEncoder()}


LabelEncoder()

In [25]:
cat_dims = [int(len(encoders[col].classes_)) for col in cat_cols]
emb_dims = [min(50, (n + 1) // 2) for n in cat_dims]          # common embedding rule of thumb
print(f'cat_dims: {cat_dims}')
print(f'emb_dims_ideas 1: {emb_dims}')

cat_dims: [3, 2, 3, 8, 3]
emb_dims_ideas 1: [2, 1, 2, 4, 2]


In [26]:
encoded_categorical_cols = [
    'Type_Operation_encoded',
    'RURAL_URBAN_encoded',
    'Terrain_Type_encoded',
    'MedianType_encoded',
    'AccessControl_encoded',
    #'CO_NAME_encoded'
]

for col in encoded_categorical_cols:
    if isinstance(pyg_data[col], np.ndarray):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)
    elif isinstance(pyg_data[col], list):
        pyg_data[col] = torch.tensor(pyg_data[col], dtype=torch.long)

In [27]:
from sklearn.model_selection import train_test_split

# nodes with measured AADT
has_aadt_mask = pyg_data.y.squeeze() > 0
y_nonzero = pyg_data.y.squeeze()[has_aadt_mask]

# 1st percentile cutoff
percentile_1 = torch.quantile(y_nonzero, 0.01) #1th percentile

# filter labeled_indices using 1st percentile threshold
labeled_indices = (pyg_data.y.squeeze() > percentile_1).nonzero(as_tuple=True)[0]

# Train/Val/Test split from labeled nodes only (no outliers)
train_idx, temp_idx = train_test_split(labeled_indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# boolean masks (all nodes) - initializing
num_nodes = pyg_data.num_nodes
pyg_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
pyg_data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# train/validation/test masks
pyg_data.train_mask[train_idx] = True
pyg_data.val_mask[val_idx] = True
pyg_data.test_mask[test_idx] = True

## Edge attr

In [28]:
# ============================================================
# Compute multi-hop edges with hop distances
# ============================================================

def build_multihop_edges(edge_index, num_nodes, max_k=3):
    A = to_scipy_sparse_matrix(edge_index, num_nodes=num_nodes)
    A = (A > 0).astype(float)
    
    all_edges = []
    all_hops = []
    
    seen = A.copy()
    
    # 1-hop (original)
    rows, cols = A.nonzero()
    all_edges.append(torch.stack([torch.tensor(rows), torch.tensor(cols)], dim=0))
    all_hops.append(torch.ones(len(rows)))
    
    Ak = A.copy()
    for k in range(2, max_k + 1):
        Ak = Ak @ A
        new_connections = (Ak > 0).astype(float)
        new_connections.setdiag(0)
        new_connections = new_connections - seen
        new_connections = (new_connections > 0).astype(float)
        
        rows, cols = new_connections.nonzero()
        if len(rows) > 0:
            all_edges.append(torch.stack([torch.tensor(rows), torch.tensor(cols)], dim=0))
            all_hops.append(torch.full((len(rows),), k, dtype=torch.float))
        
        seen = (seen + new_connections > 0).astype(float)
    
    edge_index_multihop = torch.cat(all_edges, dim=1).long()
    hop_distances = torch.cat(all_hops)
    
    return edge_index_multihop, hop_distances

In [29]:
# ============================================================
#  Compute functional class difference per edge
# ============================================================

def compute_functional_diff(edge_index, node_functional_classes):
    """
    node_functional_classes: tensor of shape [num_nodes] 
    e.g. each node has a functional class like 0, 1, 2, 3...
    """
    src = edge_index[0]
    dst = edge_index[1]
    delta_f = torch.abs(node_functional_classes[src] - node_functional_classes[dst]).float()
    return delta_f

In [30]:
# ============================================================
#  2D Sinusoidal Edge Encoder
# ============================================================

class SinusoidalEdgeEncoder(torch.nn.Module):
    def __init__(self, d_model=64, max_freq_base=10000.0):
        """
        d_model: total encoding dimension (split in half: hop + functional)
        """
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even (split between hop and func)"
        
        self.d_half = d_model // 2
        
        # precompute frequency bands for each half
        # each half uses sin/cos pairs, so d_half must also be even
        assert self.d_half % 2 == 0, "d_model must be divisible by 4"
        
        n_freqs = self.d_half // 2  # number of sin/cos pairs per half
        freq_indices = torch.arange(n_freqs).float()
        
        # frequencies: 1/10000^(2i/d_half)
        freqs = 1.0 / (max_freq_base ** (2 * freq_indices / self.d_half))
        self.register_buffer('freqs', freqs)  # [n_freqs]
    
    def _encode(self, values):
        """
        values: [num_edges] — scalar per edge
        returns: [num_edges, d_half]
        """
        # values: [num_edges, 1] * freqs: [n_freqs] -> [num_edges, n_freqs]
        angles = values.unsqueeze(-1) * self.freqs.unsqueeze(0)
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)  # [num_edges, d_half]
    
    def forward(self, hop_distances, delta_f):
        """
        hop_distances: [num_edges]
        delta_f: [num_edges]
        returns: [num_edges, d_model]
        """
        pe_hop = self._encode(hop_distances)    # [num_edges, d_half]
        pe_func = self._encode(delta_f)          # [num_edges, d_half]
        return torch.cat([pe_hop, pe_func], dim=-1)  # [num_edges, d_model]


In [ ]:
# ============================================================
#  Put it all together
# ============================================================

def prepare_graph_data(pyg_data, node_functional_classes, max_k=2, edge_dim=64):
    """
    pyg_data: your existing PyG Data object with .x and .edge_index
    node_functional_classes: tensor [num_nodes] with functional class per node
    max_k: how many hops to add
    edge_dim: dimension of the sinusoidal edge encoding
    
    Returns: new PyG Data object with multi-hop edges and edge_attr
    """
    pyg_data.old_edge_index = pyg_data.edge_index.clone() #clone and keep original index
    
    # multi-hop edges
    edge_index_new, hop_distances = build_multihop_edges(
        pyg_data.edge_index, pyg_data.num_nodes, max_k=max_k
    )
    
    # functional class difference
    delta_f = compute_functional_diff(edge_index_new, node_functional_classes)
    
    # encode
    encoder = SinusoidalEdgeEncoder(d_model=edge_dim)
    edge_attr = encoder(hop_distances, delta_f)
    
    # new Data object
    pyg_data.edge_index = edge_index_new
    pyg_data.edge_attr = edge_attr
    
    # store these in case you need them later
    pyg_data.hop_distances = hop_distances
    pyg_data.delta_f = delta_f
    
    return pyg_data, encoder

In [32]:
node_functional_classes = torch.tensor(pyg_data.F_SYSTEM) # shape [num_nodes]

# build everything
new_data, encoder = prepare_graph_data(
    pyg_data, 
    node_functional_classes, 
    max_k=3,      # start with 2-hop, check edge count
    edge_dim=64   # encoding vector size
)

print(f"Original edges: {pyg_data.old_edge_index.shape[1]}")
print(f"Multi-hop edges: {new_data.edge_index.shape[1]}")
print(f"Edge attr shape: {new_data.edge_attr.shape}")  # [num_edges, 64]

Original edges: 1361537
Multi-hop edges: 6765030
Edge attr shape: torch.Size([6765030, 64])


In [33]:
for key, value in new_data:
    try:
        if isinstance(value, list):
            new_data[key] = torch.tensor(value)
    except:
        pass

## Modeling

In [34]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, summary
from torch_geometric.loader import NeighborLoader
torch.manual_seed(42)

In [35]:
def mean_absolute_percentage_error_(y_true, y_pred):
    # Avoid division by zero
    epsilon = 1e-10
    return torch.mean(torch.abs((y_true - y_pred) / (y_true + epsilon))) * 100

In [36]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [37]:
class WeightedMAPELoss(nn.Module):
    def __init__(self, epsilon=1.0):
        """
        epsilon: floor to prevent division by zero.
        Set to 1.0 for AADT since traffic counts are integers.
        """
        super().__init__()
        self.epsilon = epsilon

    def forward(self, pred, true):
        denom = torch.clamp(true.abs(), min=self.epsilon)
        return (torch.abs(pred - true) / denom).mean()

In [38]:
class BlendedLoss(nn.Module):
    def __init__(self, alpha=0.7, epsilon=1.0):
        """
        alpha: weight on MAPE (0.7 = mostly MAPE, 0.3 = some RMSE)
        """
        super().__init__()
        self.alpha = alpha
        self.epsilon = epsilon

    def forward(self, pred, true):
        # MAPE component
        denom = torch.clamp(true.abs(), min=self.epsilon)
        mape = (torch.abs(pred - true) / denom).mean()

        # RMSE component (normalized so it's on similar scale)
        rmse = torch.sqrt(torch.mean((pred - true) ** 2)) / true.abs().mean()

        return self.alpha * mape + (1 - self.alpha) * rmse
    
# Instead of:
#loss_fn = nn.MSELoss()

# Use:
#loss_fn = BlendedLoss(alpha=0.7)  # 70% MAPE, 30% normalized RMSE

In [39]:
import copy

def train_and_evaluate(data, model, device, epochs=2000, lr=1e-3, batch_size=256,
                       save_path='best_model_checkpoint.pt'):

    print(f"Training device: {device}")

    train_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.train_mask,
        shuffle=True,
    )

    val_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.val_mask,
        shuffle=False,
    )

    test_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.test_mask,
        shuffle=False,
    )

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    # Track best of both metrics
    best_val_rmse = float('inf')
    best_val_mape = float('inf')
    rmse_at_best_mape = float('inf')

    for epoch in range(1, epochs + 1):

        # -----------------------------------------------------------------
        # Training step
        # -----------------------------------------------------------------
        model.train()
        total_train_loss = 0
        total_train_nodes = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            seed_out = out[:batch.batch_size]
            seed_y = batch.y[:batch.batch_size]

            loss = loss_fn(seed_out.squeeze(), seed_y.squeeze())
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * batch.batch_size
            total_train_nodes += batch.batch_size

        # -----------------------------------------------------------------
        # Evaluation step
        # -----------------------------------------------------------------
        model.eval()
        with torch.no_grad():

            def evaluate_loader(loader):
                all_preds = []
                all_trues = []
                for batch in loader:
                    batch = batch.to(device)
                    out = model(batch)
                    seed_out = out[:batch.batch_size]
                    seed_y = batch.y[:batch.batch_size]
                    all_preds.append(seed_out.squeeze())
                    all_trues.append(seed_y.squeeze())
                return torch.cat(all_preds), torch.cat(all_trues)

            train_pred, train_true = evaluate_loader(train_loader)
            val_pred,   val_true   = evaluate_loader(val_loader)
            test_pred,  test_true  = evaluate_loader(test_loader)

            train_rmse = torch.sqrt(loss_fn(train_pred, train_true)).item()
            val_rmse   = torch.sqrt(loss_fn(val_pred,   val_true)).item()
            test_rmse  = torch.sqrt(loss_fn(test_pred,  test_true)).item()

            train_mape = mean_absolute_percentage_error_(train_true, train_pred).item()
            val_mape   = mean_absolute_percentage_error_(val_true,   val_pred).item()
            test_mape  = mean_absolute_percentage_error_(test_true,  test_pred).item()

        # -----------------------------------------------------------------
        # Track best RMSE (for reference)
        # -----------------------------------------------------------------
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse

        # -----------------------------------------------------------------
        # Checkpoint on MAPE, but only if RMSE hasn't blown up
        # -----------------------------------------------------------------
        if val_mape < best_val_mape and val_rmse < best_val_rmse * 1.15:
            best_val_mape = val_mape
            rmse_at_best_mape = val_rmse
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, save_path)

        # -----------------------------------------------------------------
        # Progress logging
        # -----------------------------------------------------------------
        if epoch % 50 == 0:
            print(f"\nProgress @ Epoch {epoch}:")
            print(f"  Best Val MAPE so far: {best_val_mape:.2f}%  (RMSE at that point: {rmse_at_best_mape:.4f})")
            print(f"  Best Val RMSE so far: {best_val_rmse:.4f}\n")
            print(f"  Train RMSE: {train_rmse:.4f}  |  Train MAPE: {train_mape:.2f}%")
            print(f"  Val RMSE:   {val_rmse:.4f}  |  Val MAPE:   {val_mape:.2f}%")
            print(f"  Test RMSE:  {test_rmse:.4f}  |  Test MAPE:  {test_mape:.2f}%\n")

    print(f"\nBest model saved — Val MAPE: {best_val_mape:.2f}% | RMSE: {rmse_at_best_mape:.4f}")

    return best_val_mape, rmse_at_best_mape

In [40]:
import copy

def train_and_evaluate_es(data, model, device, epochs=2000, lr=1e-3, batch_size=256,
                       save_path='best_model_checkpoint.pt', patience=200):
    """
    Mini-batch training with:
    - MAPE-based checkpointing (with RMSE guard)
    - Early stopping when val MAPE doesn't improve for `patience` epochs
    - LR scheduler that halves LR when val MAPE plateaus
    """

    print(f"Training device: {device}")

    train_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.train_mask,
        shuffle=True,
    )

    val_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.val_mask,
        shuffle=False,
    )

    test_loader = NeighborLoader(
        data,
        num_neighbors=[-1, -1, -1],
        batch_size=batch_size,
        input_nodes=data.test_mask,
        shuffle=False,
    )

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=50, 
    )
    loss_fn = nn.MSELoss()

    # Tracking
    best_val_rmse = float('inf')
    best_val_mape = float('inf')
    rmse_at_best_mape = float('inf')
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):

        # -----------------------------------------------------------------
        # Training step
        # -----------------------------------------------------------------
        model.train()
        total_train_loss = 0
        total_train_nodes = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            out = model(batch)
            seed_out = out[:batch.batch_size]
            seed_y = batch.y[:batch.batch_size]

            loss = loss_fn(seed_out.squeeze(), seed_y.squeeze())
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * batch.batch_size
            total_train_nodes += batch.batch_size

        # -----------------------------------------------------------------
        # Evaluation step
        # -----------------------------------------------------------------
        model.eval()
        with torch.no_grad():

            def evaluate_loader(loader):
                all_preds = []
                all_trues = []
                for batch in loader:
                    batch = batch.to(device)
                    out = model(batch)
                    seed_out = out[:batch.batch_size]
                    seed_y = batch.y[:batch.batch_size]
                    all_preds.append(seed_out.squeeze())
                    all_trues.append(seed_y.squeeze())
                return torch.cat(all_preds), torch.cat(all_trues)

            train_pred, train_true = evaluate_loader(train_loader)
            val_pred,   val_true   = evaluate_loader(val_loader)
            test_pred,  test_true  = evaluate_loader(test_loader)

            train_rmse = torch.sqrt(loss_fn(train_pred, train_true)).item()
            val_rmse   = torch.sqrt(loss_fn(val_pred,   val_true)).item()
            test_rmse  = torch.sqrt(loss_fn(test_pred,  test_true)).item()

            train_mape = mean_absolute_percentage_error_(train_true, train_pred).item()
            val_mape   = mean_absolute_percentage_error_(val_true,   val_pred).item()
            test_mape  = mean_absolute_percentage_error_(test_true,  test_pred).item()

        # Step the scheduler based on val MAPE
        scheduler.step(val_mape)

        # -----------------------------------------------------------------
        # Track best RMSE (for the guard)
        # -----------------------------------------------------------------
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse

        # -----------------------------------------------------------------
        # Checkpoint on MAPE, guarded by RMSE
        # -----------------------------------------------------------------
        if val_mape < best_val_mape and val_rmse < best_val_rmse * 1.15:
            best_val_mape = val_mape
            rmse_at_best_mape = val_rmse
            best_model_state = copy.deepcopy(model.state_dict())
            torch.save(best_model_state, save_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        # -----------------------------------------------------------------
        # Early stopping
        # -----------------------------------------------------------------
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping at epoch {epoch} — no MAPE improvement for {patience} epochs")
            print(f"Best Val MAPE: {best_val_mape:.2f}% | RMSE: {rmse_at_best_mape:.4f}")
            break

        # -----------------------------------------------------------------
        # Progress logging
        # -----------------------------------------------------------------
        if epoch % 50 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"\nProgress @ Epoch {epoch} (LR: {current_lr:.1e}, no improvement: {epochs_without_improvement}):")
            print(f"  Best Val MAPE so far: {best_val_mape:.2f}%  (RMSE at that point: {rmse_at_best_mape:.4f})")
            print(f"  Best Val RMSE so far: {best_val_rmse:.4f}\n")
            print(f"  Train RMSE: {train_rmse:.4f}  |  Train MAPE: {train_mape:.2f}%")
            print(f"  Val RMSE:   {val_rmse:.4f}  |  Val MAPE:   {val_mape:.2f}%")
            print(f"  Test RMSE:  {test_rmse:.4f}  |  Test MAPE:  {test_mape:.2f}%\n")

    print(f"\nBest model saved — Val MAPE: {best_val_mape:.2f}% | RMSE: {rmse_at_best_mape:.4f}")

    return best_val_mape, rmse_at_best_mape

In [41]:
import copy
import torch
from torch import Tensor
from torch_geometric.nn.conv import MessagePassing


class DirectionalGNNConv(torch.nn.Module):
    def __init__(
        self,
        conv: MessagePassing,
        alpha: float = 0.5,
        root_weight: bool = True,
    ):
        super().__init__()

        self.alpha = torch.nn.Parameter(torch.tensor(alpha))
        self.root_weight = root_weight

        self.conv_in = copy.deepcopy(conv)
        self.conv_out = copy.deepcopy(conv)

        if hasattr(conv, 'add_self_loops'):
            self.conv_in.add_self_loops = False
            self.conv_out.add_self_loops = False
        if hasattr(conv, 'root_weight'):
            self.conv_in.root_weight = False
            self.conv_out.root_weight = False

        if root_weight:
            self.lin = torch.nn.Linear(conv.in_channels, conv.out_channels)
        else:
            self.lin = None

        self.reset_parameters()

    def reset_parameters(self):
        self.conv_in.reset_parameters()
        self.conv_out.reset_parameters()
        if self.lin is not None:
            self.lin.reset_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, **kwargs) -> Tensor:
        alpha = torch.sigmoid(self.alpha)

        x_in = self.conv_in(x, edge_index, **kwargs)
        x_out = self.conv_out(x, edge_index.flip([0]), **kwargs)

        out = alpha * x_out + (1 - alpha) * x_in

        if self.root_weight:
            out = out + self.lin(x)

        return out

    def __repr__(self) -> str:
        alpha_val = torch.sigmoid(self.alpha).item()
        return f'{self.__class__.__name__}({self.conv_in}, alpha={alpha_val:.3f})'

In [45]:
from torch_geometric.nn import LayerNorm
class GNNModel(nn.Module):
    def __init__(
        self,
        continuous_dims,
        gnn_in_dims,
        gnn_out_dims,
        ln_in_dims,
        edge_dim=16,
        heads=4,
        dropout=0.3,
    ):
        super().__init__()
        self.dropout = dropout

        # -----------------------------------------------------------------
        # Categorical embedding layers
        # -----------------------------------------------------------------
        self.embeddings = nn.ModuleDict({
            'Type_Operation_encoded': nn.Embedding(3, 3),
            'RURAL_URBAN_encoded':    nn.Embedding(2, 2),
            'Terrain_Type_encoded':   nn.Embedding(3, 3),
            'MedianType_encoded':     nn.Embedding(8, 5),
            'AccessControl_encoded':  nn.Embedding(3, 3),
        })

        total_embed_dim = sum(emb.embedding_dim for emb in self.embeddings.values())
        input_dim = total_embed_dim + continuous_dims

        # -----------------------------------------------------------------
        # DirGNNConv-wrapped GATv2 layers
        # -----------------------------------------------------------------
        self.conv1 = DirectionalGNNConv(
            GATv2Conv(
                in_channels=input_dim,
                out_channels=gnn_in_dims,
                heads=heads,
                edge_dim=edge_dim,
                concat=True,
                dropout=dropout,
                add_self_loops=False,
            ),
            alpha=0.0,
            root_weight=False,
        )
        self.bn1 = LayerNorm(gnn_in_dims * heads, affine=True, mode='node')

        self.conv2 = DirectionalGNNConv(
            GATv2Conv(
                in_channels=gnn_in_dims * heads,
                out_channels=gnn_in_dims,
                heads=heads,
                edge_dim=edge_dim,
                concat=True,
                dropout=dropout,
                add_self_loops=False,
            ),
            alpha=0.0,
            root_weight=False,
        )
        self.bn2 = LayerNorm(gnn_in_dims * heads, affine=True, mode='node')

        self.conv3 = DirectionalGNNConv(
            GATv2Conv(
                in_channels=gnn_in_dims * heads,
                out_channels=gnn_in_dims,
                heads=1,
                edge_dim=edge_dim,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),
            alpha=0.0,
            root_weight=False,
        )
        self.bn3 = LayerNorm(gnn_in_dims, affine=True, mode='node')

        # -----------------------------------------------------------------
        # Residual skip connections
        # -----------------------------------------------------------------
        self.skip1 = nn.Linear(input_dim, gnn_in_dims * heads)
        self.skip2 = nn.Linear(gnn_in_dims * heads, gnn_in_dims * heads)
        self.skip3 = nn.Linear(gnn_in_dims * heads, gnn_in_dims)

        # -----------------------------------------------------------------
        # MLP regression head
        # -----------------------------------------------------------------
        self.regressor = nn.Sequential(
            nn.Linear(gnn_in_dims, ln_in_dims),
            nn.ReLU(),
            nn.Linear(ln_in_dims, 1),
        )

    def forward(self, data):
        # -----------------------------------------------------------------
        #  Build node feature matrix from embeddings + continuous features
        # -----------------------------------------------------------------
        embedded = []
        for col in self.embeddings:
            x_cat = data[col].long()
            embedded.append(self.embeddings[col](x_cat))
        x_cat_embed = torch.cat(embedded, dim=1)

        x_cont = data.x
        x = torch.cat([x_cat_embed, x_cont], dim=1)

        # -----------------------------------------------------------------
        #  DirGNNConv message passing
        # -----------------------------------------------------------------
        x1 = self.conv1(x, data.edge_index, edge_attr=data.edge_attr)
        x1 = self.bn1(x1)
        x1 = F.relu(x1) + self.skip1(x)
        x1 = F.dropout(x1, p=self.dropout, training=self.training)

        x2 = self.conv2(x1, data.edge_index, edge_attr=data.edge_attr)
        x2 = self.bn2(x2)
        x2 = F.relu(x2) + self.skip2(x1)
        x2 = F.dropout(x2, p=self.dropout, training=self.training)

        x3 = self.conv3(x2, data.edge_index, edge_attr=data.edge_attr)
        x3 = self.bn3(x3)
        x3 = x3 + self.skip3(x2)

        return torch.abs(self.regressor(x3))

In [46]:
new_data.x.shape[1], new_data.x.size(1)

(18, 18)

In [47]:
continuous_dims = new_data.x.size(1)
model = GNNModel(
    continuous_dims = continuous_dims,
    gnn_in_dims = 256,
    gnn_out_dims = 128,
    ln_in_dims = 64,
    edge_dim = new_data.edge_attr.shape[1],
    heads = 4,
    dropout=0.1,
)

In [48]:
from torch_geometric.data import Data

keep = [
    'x', 'y', 'edge_index', 'edge_attr',
    'train_mask', 'val_mask', 'test_mask',
    'Type_Operation_encoded', 'RURAL_URBAN_encoded',
    'Terrain_Type_encoded', 'MedianType_encoded', 'AccessControl_encoded',
]

model_data = Data(**{key: new_data[key] for key in keep})

# Make sure everything is a tensor
for key, value in model_data:
    if isinstance(value, list):
        model_data[key] = torch.tensor(value)

print(model_data)

Data(x=[404070, 18], edge_index=[2, 6765030], edge_attr=[6765030, 64], y=[404070, 1], train_mask=[404070], val_mask=[404070], test_mask=[404070], Type_Operation_encoded=[404070], RURAL_URBAN_encoded=[404070], Terrain_Type_encoded=[404070], MedianType_encoded=[404070], AccessControl_encoded=[404070])


In [49]:
print(model)

GNNModel(
  (embeddings): ModuleDict(
    (Type_Operation_encoded): Embedding(3, 3)
    (RURAL_URBAN_encoded): Embedding(2, 2)
    (Terrain_Type_encoded): Embedding(3, 3)
    (MedianType_encoded): Embedding(8, 5)
    (AccessControl_encoded): Embedding(3, 3)
  )
  (conv1): DirectionalGNNConv(GATv2Conv(34, 256, heads=4), alpha=0.500)
  (bn1): LayerNorm(1024, affine=True, mode=node)
  (conv2): DirectionalGNNConv(GATv2Conv(1024, 256, heads=4), alpha=0.500)
  (bn2): LayerNorm(1024, affine=True, mode=node)
  (conv3): DirectionalGNNConv(GATv2Conv(1024, 256, heads=1), alpha=0.500)
  (bn3): LayerNorm(256, affine=True, mode=node)
  (skip1): Linear(in_features=34, out_features=1024, bias=True)
  (skip2): Linear(in_features=1024, out_features=1024, bias=True)
  (skip3): Linear(in_features=1024, out_features=256, bias=True)
  (regressor): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [50]:
save_path = 'checkpoints/'
batch_size = 256

In [51]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
train_and_evaluate_es(model_data, model, device, epochs=4000, lr=1e-3, batch_size=batch_size,patience=200,
                       save_path=save_path + 'dir_gat_update_cp_nbc_batch_256.pt')

Training device: cuda


## Final scores

In [ ]:
# Load best model
model.load_state_dict(torch.load(save_path + 'dir_gat_update_cp_nbc_batch_256.pt'))
model = model.to(device)
model.eval()

In [ ]:
# Create loaders
train_loader = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.train_mask, shuffle=False)

val_loader   = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.val_mask, shuffle=False)

test_loader  = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.test_mask, shuffle=False)

In [ ]:
from sklearn.metrics import r2_score

def evaluate_loader(loader):
    all_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            all_preds.append(out[:batch.batch_size].squeeze())
            all_trues.append(batch.y[:batch.batch_size].squeeze())
    return torch.cat(all_preds), torch.cat(all_trues)

In [ ]:
train_pred, train_true = evaluate_loader(train_loader)
val_pred,   val_true   = evaluate_loader(val_loader)
test_pred,  test_true  = evaluate_loader(test_loader)

for name, pred, true in [('Train', train_pred, train_true),
                          ('Val',   val_pred,   val_true),
                          ('Test',  test_pred,  test_true)]:
    rmse = torch.sqrt(loss_fn(pred, true)).item()
    mape = mean_absolute_percentage_error_(true, pred).item()
    mae  = torch.mean(torch.abs(pred - true)).item()
    r2   = r2_score(true.cpu().numpy(), pred.cpu().numpy())
    print(f"{name:5s} — RMSE: {rmse:.4f} | MAE: {mae:.4f} | MAPE: {mape:.2f}% | R²: {r2:.4f}")

##  --- Get predictions for all splits ---

In [ ]:

train_loader = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.train_mask, shuffle=False)
val_loader   = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.val_mask, shuffle=False)
test_loader  = NeighborLoader(model_data, num_neighbors=[-1,-1,-1], batch_size=256,
                               input_nodes=model_data.test_mask, shuffle=False)

def get_predictions(loader):
    all_preds = []
    all_trues = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            all_preds.append(out[:batch.batch_size].squeeze().cpu())
            all_trues.append(batch.y[:batch.batch_size].squeeze().cpu())
    return torch.cat(all_preds).numpy(), torch.cat(all_trues).numpy()

train_pred, train_true = get_predictions(train_loader)
val_pred,   val_true   = get_predictions(val_loader)
test_pred,  test_true  = get_predictions(test_loader)

# --- Define bins ---
bin_edges = [
    0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000,
    5500, 6000, 6500, 7000, 7500, 8000, 8500, 9000, 9500, 10000,
    20000, 35000, 55000, 85000, 125000, np.inf
]
bin_labels = [
    "< 500", "500 - 1000", "1000 - 1500", "1500 - 2000", "2000 - 2500", "2500 - 3000",
    "3000 - 3500", "3500 - 4000", "4000 - 4500", "4500 - 5000", "5000 - 5500", "5500 - 6000",
    "6000 - 6500", "6500 - 7000", "7000 - 7500", "7500 - 8000", "8000 - 8500", "8500 - 9000",
    "9000 - 9500", "9500 - 10000", "10000 - 20000",
    "20000 - 35000", "35000 - 55000", "55000 - 85000", "85000 - 125000", ">125000"
]

# --- Metrics function ---
def bin_metrics(group):
    t = group["y_true"].values
    p = group["y_pred"].values
    n = len(t)
    rmse = np.sqrt(np.mean((p - t) ** 2))
    mae  = np.mean(np.abs(p - t))
    mape = np.mean(np.abs((t - p) / np.clip(t, 1, None))) * 100
    r2   = r2_score(t, p) if n > 1 else np.nan
    return pd.Series({"N": n, "RMSE": rmse, "MAE": mae, "MAPE": mape, "R2": r2})

# --- Run for each split + combined ---
splits = {
    "Train": (train_pred, train_true),
    "Val":   (val_pred,   val_true),
    "Test":  (test_pred,  test_true),
    "All":   (np.concatenate([train_pred, val_pred, test_pred]),
              np.concatenate([train_true, val_true, test_true])),
}

for split_name, (pred, true) in splits.items():
    # Filter non-zero
    mask = true > 0
    df = pd.DataFrame({"y_true": true[mask], "y_pred": pred[mask]})
    df["aadt_bin"] = pd.cut(df["y_true"], bins=bin_edges, labels=bin_labels,
                            right=False, include_lowest=True)

    results = df.groupby("aadt_bin", observed=False).apply(bin_metrics)

    # Add overall row
    overall = bin_metrics(df)
    overall.name = "OVERALL"
    results = pd.concat([results, overall.to_frame().T])

    print(f"\n{'='*80}")
    print(f"  {split_name} Set Metrics by AADT Bin")
    print(f"{'='*80}")
    print(results.to_string())